In [1]:
import os
import subprocess
import warnings

from datetime import datetime
from pathlib import Path

import kagglehub
import lightgbm as lgb
import xgboost as xgb
import catboost as ctb

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report

RANDOM_STATE = 42
N_SPLIT = 5
MODELS_TO_RUN = ('lightgbm', 'catboost', 'xgboost')

def cuda_gpu_avilable() -> bool:
    if os.environ.get('CUDA_VISIBLE_DEVICES') in {'', -1}:
        return False
    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
            check=True, capture_output=True, text=True, timeout=5
        )
        return bool(result.stdout.strip())
    except Exception:
        return False

USE_GPU = cuda_gpu_avilable()
LGBM_DEVICE_PARAMS = {'device_type': 'gpu'} if USE_GPU else {}
CTB_DEVICE_PARAMS = {'task_type': 'GPU', 'devices': '0'} if USE_GPU else {}
XGB_DEVICE_PARAMS = {'device': 'cuda'} if USE_GPU else {}
print(f'CUDA GPU available: {USE_GPU}')

DATA_DIR = Path(kagglehub.competition_download('playground-series-s6e9'))
OUTPUT_DIR = Path(f'output/{int(datetime.now().timestamp())}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

TARGET = 'Will_Buy_EV'
ID_COLUMN = 'id'
assert train.drop(columns=TARGET).columns.tolist() == test.columns.tolist()
assert sample_submission[ID_COLUMN].equals(test[ID_COLUMN])
print(f'Train: {train.shape} | Test: {test.shape}')
print(f'Target distribution:\n{train[TARGET].value_counts(normalize=True)}')
warnings.filterwarnings('ignore')

CUDA GPU available: False
Train: (668665, 15) | Test: (286571, 14)
Target distribution:
Will_Buy_EV
No     0.825355
Yes    0.174645
Name: proportion, dtype: float64


In [2]:
NUMERIC_COLUMNS = train.drop([ID_COLUMN, TARGET], axis=1).select_dtypes('number').columns.to_list()
CATEGORICAL_COLUMNS = train.drop([ID_COLUMN, TARGET], axis=1).select_dtypes('str').columns.to_list()

def build_features(df: pd.DataFrame) -> pd.DataFrame:
    ndf = df.drop(columns=[ID_COLUMN, TARGET], errors='ignore').copy()
    return ndf

X_train = build_features(train)
X_test = build_features(test)
target_map = {'Yes': 1, 'No': 0}
y_train = train[TARGET].map(target_map).astype('int8')

assert X_train.columns.to_list() == X_test.columns.to_list()
assert X_train.shape[0] == len(y_train)

print(f'Numberical Columns: {NUMERIC_COLUMNS}')
print(f'Categorical columns: {CATEGORICAL_COLUMNS}')
print(f'Model features: {X_train.shape}, y_train len: {len(y_train)}')

Numberical Columns: ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level']
Categorical columns: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']
Model features: (668665, 13), y_train len: 668665


In [3]:
def make_lightgbm_input(df: pd.DataFrame, test_df: pd.DataFrame):
    train_input, test_input = df.copy(), test_df.copy()
    for col in CATEGORICAL_COLUMNS:
        categories = pd.Index(pd.concat([train_input[col], test_input[col]], ignore_index=True).unique())
        train_input[col] = pd.Categorical(train_input[col], categories=categories)
        test_input[col] = pd.Categorical(test_input[col], categories=categories)
    return train_input, test_input

def make_catboost_input(df: pd.DataFrame, test_df: pd.DataFrame):
    train_input, test_input = df.copy(), test_df.copy()
    for col in CATEGORICAL_COLUMNS:
        train_input[col] = train_input[col].astype(str)
        test_input[col] = test_input[col].astype(str)
    return train_input, test_input

x_train_lgbm, x_test_lgbm = make_lightgbm_input(X_train, X_test)
x_train_cat, x_test_cat = make_lightgbm_input(X_train, X_test)

# One-hot encoding is used only for the optional XGBoost model.
xgb_combined = pd.get_dummies(pd.concat([X_train, X_test], axis=0), columns=CATEGORICAL_COLUMNS, dtype=np.int8)
x_train_xgb, x_test_xgb = xgb_combined.iloc[:len(X_train)], xgb_combined.iloc[len(X_train):]

folds = list(StratifiedKFold(n_splits=N_SPLIT, shuffle=True, random_state=RANDOM_STATE).split(X_train, y_train))

In [4]:
def run_lightgbm_cv(x_train, x_test, y, folds, params={}):
    oof, test_prediction = np.zeros(len(x_train)), np.zeros(len(x_test))
    for fold, (train_index, vaild_index) in enumerate(folds, start=1):
        model_params = {
            'objective': 'binary', 'n_estimators': 5000, 'learning_rate': 0.33, 'num_leaves': 63,
            'min_child_samples': 80, 'subsample': 0.85, 'colsample_bytree': 0.85, 'reg_lambda': 2.0,
            'random_state': RANDOM_STATE+fold, 'n_jobs': -1, 'verbosity': -1, **LGBM_DEVICE_PARAMS, **params,
        }
        model = lgb.LGBMClassifier(**model_params)
        model.fit(
            x_train.iloc[train_index], y.iloc[train_index],
            eval_set=[(x_train.iloc[vaild_index], y.iloc[vaild_index])], eval_metric='auc',
            categorical_feature=CATEGORICAL_COLUMNS,
            callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(250)],
        )
        oof[vaild_index] = model.predict_proba(x_train.iloc[vaild_index])[:, 1]
        test_prediction += model.predict_proba(x_test)[:, 1] / N_SPLIT
        print(f'LightGBM fold {fold}: {roc_auc_score(y.iloc[vaild_index], oof[vaild_index]):.6f} | trees: {model.best_iteration_}')
    print(f'LightGBM OOF AUC: {roc_auc_score(y, oof):.6f}')
    return oof, test_prediction

def run_catboost_cv(x_train, x_test, y, folds, params={}):
    oof, test_prediction = np.zeros(len(x_train)), np.zeros(len(x_test))
    for fold, (train_index, valid_index) in enumerate(folds, start=1):
        model_params = {
            'loss_function': 'Logloss', 'eval_metric': 'AUC', 'iterations': 5000, 'learning_rate': 0.083,
            'depth': 8, 'l2_leaf_reg': 5.0, 'random_seed': RANDOM_STATE + fold,
            'verbose': 250, 'allow_writing_files': False, **CTB_DEVICE_PARAMS, **params
        }
        model = ctb.CatBoostClassifier(**model_params)
        model.fit(
            x_train.iloc[train_index], y.iloc[train_index], cat_features=CATEGORICAL_COLUMNS,
            eval_set=(x_train.iloc[valid_index], y.iloc[valid_index]), early_stopping_rounds=200,
        )
        oof[valid_index] = model.predict_proba(x_train.iloc[valid_index])[:, 1]
        test_prediction += model.predict_proba(x_test)[:, 1]/N_SPLIT
        print(f'CatBoost fold {fold}: {roc_auc_score(y.iloc[valid_index], oof[valid_index]):.6f} | trees: {model.best_iteration_}')
    print(f'CatBoost OOF AUC: {roc_auc_score(y, oof):.6f}')
    return oof, test_prediction

def run_xgboost_cv(x_train, x_test, y, folds, params={}):
    oof, test_prediction = np.zeros(len(x_train)), np.zeros(len(x_test))
    for fold, (train_index, valid_index) in enumerate(folds, start=1):
        model_params = {
            'objective': 'binary:logistic', 'eval_metric': 'auc', 'n_estimators': 5000,
            'learning_rate': 0.03, 'max_depth': 8, 'min_child_weight': 8,
            'subsample': 0.85, 'colsample_bytree': 0.85, 'reg_lambda': 3.0,
            'tree_method': 'hist', 'early_stopping_rounds': 200,
            'random_state': RANDOM_STATE + fold, 'n_jobs': -1, **XGB_DEVICE_PARAMS, **params,
        }
        model = xgb.XGBClassifier(**model_params)
        model.fit(
            x_train.iloc[train_index], y.iloc[train_index],
            eval_set=[(x_train.iloc[valid_index], y.iloc[valid_index])], verbose=250
        )
        oof[valid_index] = model.predict_proba(x_train.iloc[valid_index])[:, 1]
        test_prediction += model.predict_proba(x_test)[:, 1] / N_SPLIT
        print(f'XGBoost fold {fold}: {roc_auc_score(y.iloc[valid_index], oof[valid_index]):.6f} | trees: {model.best_iteration}')
    print(f'XGBoost OOF AUC: {roc_auc_score(y, oof):.6f}')
    return oof, test_prediction

def oof_and_test_predictions(lgbm_params={}, ctb_params={}, xgb_params={}):
    oof_predictions, test_predictions = {}, {}

    if 'lightgbm' in MODELS_TO_RUN:
        oof_predictions['lightgbm'], test_predictions['lightgbm'] = run_lightgbm_cv(x_train_lgbm, x_test_lgbm, y_train, folds, params=lgbm_params)
    if 'catboost' in MODELS_TO_RUN:
        oof_predictions['catboost'], test_predictions['catboost'] = run_catboost_cv(x_train_cat, x_test_cat, y_train, folds, params=ctb_params)
    if 'xgboost' in MODELS_TO_RUN:
        oof_predictions['xgboost'], test_predictions['xgboost'] = run_xgboost_cv(x_train_xgb, x_test_xgb, y_train, folds, params=xgb_params)

    scores = pd.Series({name: roc_auc_score(y_train, prediction) for name, prediction in oof_predictions.items()})
    display(scores.sort_values(ascending=False).to_frame('OOF ROC AUC'))
    return oof_predictions, test_predictions


In [ ]:
def normalized_rank(values):
    return pd.Series(values).rank(method='average').to_numpy() / (len(values) + 1)

def select_blend(oof_predictions, y, step=0.05):
    """Find blend weights from OOF predictions only; supports up to three models."""
    names = list(oof_predictions)
    if len(names) == 1:
        return names, np.array([1.0]), False, roc_auc_score(y, oof_predictions[names[0]])
    if len(names) > 3:
        raise ValueError('Blend at most three models at a time.')
    candidates, grid = [], np.arange(0, 1 + step / 2, step)
    for use_rank in (False, True):
        arrays = [normalized_rank(oof_predictions[name]) if use_rank else oof_predictions[name] for name in names]
        weights = ([(w, 1 - w) for w in grid] if len(names) == 2 else
                   [(a, b, 1 - a - b) for a in grid for b in grid if a + b <= 1])
        for weight_set in weights:
            prediction = sum(weight * values for weight, values in zip(weight_set, arrays))
            candidates.append((roc_auc_score(y, prediction), np.array(weight_set), use_rank))
    score, weights, use_rank = max(candidates, key=lambda item: item[0])
    return names, weights, use_rank, score

def blend_oof_based_on_weight_or_rank(exp_no:str, lgbm_params={}, ctb_params={}, xgb_params={}):
    oof_predictions, test_predictions = oof_and_test_predictions(
        lgbm_params=lgbm_params,
        ctb_params=ctb_params,
        xgb_params=xgb_params
    )

    OOF_OUPUT_DIR = Path(OUTPUT_DIR / f'{exp_no}')
    OOF_OUPUT_DIR.mkdir(parents=True, exist_ok=True)

    pd.DataFrame({TARGET: y_train, **oof_predictions}).to_csv(OOF_OUPUT_DIR / 'oof_predictions.csv', index=False)
    pd.DataFrame(test_predictions).to_csv(OOF_OUPUT_DIR / 'test_predictions.csv', index=False)

    model_names, blend_weights, use_rank_blend, blend_auc = select_blend(oof_predictions, y_train)
    print(f'Best OOF blend AUC: {blend_auc:.6f}')
    print(f'Models: {model_names} | Weights: {blend_weights} | Rank blend: {use_rank_blend}')

    blend_components = [normalized_rank(test_predictions[name]) if use_rank_blend else test_predictions[name] for name in model_names]
    final_prediction = sum(weight * prediction for weight, prediction in zip(blend_weights, blend_components))

    submission = pd.DataFrame({ID_COLUMN: test[ID_COLUMN], TARGET: final_prediction})
    assert submission.shape == sample_submission.shape
    assert submission[ID_COLUMN].equals(sample_submission[ID_COLUMN])
    assert submission[TARGET].between(0, 1).all()

    submission_path = OOF_OUPUT_DIR / 'submission_blend.csv'
    submission.to_csv(submission_path, index=False)
    print(f'Saved submission to: {submission_path}')
    submission.head()

# Experiment 1

In [6]:
if False:
    blend_oof_based_on_weight_or_rank('exp1')

In [7]:
import warnings
warnings.filterwarnings('ignore')

# Run only after recording the baseline above. These use three fixed folds to control
# runtime; verify each selected configuration with the full 15-fold pipeline before submission.
RUN_OPTUNA = False
OPTUNA_MODELS = ('lightgbm', 'catboost', 'xgboost')  # Remove models here to shorten a run.
N_TRIALS_PER_MODEL = 30
TUNING_FOLDS = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE).split(X_train, y_train))

if RUN_OPTUNA:
    import optuna

    def mean_fold_auc(model_factory, X):
        fold_scores = []
        for train_index, valid_index in TUNING_FOLDS:
            model = model_factory()
            X_train_fold, X_valid_fold = X.iloc[train_index], X.iloc[valid_index]
            y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]
            if isinstance(model, lgb.LGBMClassifier):
                model.fit(X_train_fold, y_train_fold,
                          eval_set=[(X_valid_fold, y_valid_fold)], eval_metric='auc',
                          categorical_feature=CATEGORICAL_COLUMNS,
                          callbacks=[lgb.early_stopping(150, verbose=False)])
            elif isinstance(model, ctb.CatBoostClassifier):
                model.fit(X_train_fold, y_train_fold, cat_features=CATEGORICAL_COLUMNS,
                          eval_set=(X_valid_fold, y_valid_fold), early_stopping_rounds=150,
                          verbose=False)
            else:
                model.fit(X_train_fold, y_train_fold,
                          eval_set=[(X_valid_fold, y_valid_fold)], verbose=False)
            prediction = model.predict_proba(X_valid_fold)[:, 1]
            fold_scores.append(roc_auc_score(y_valid_fold, prediction))
        return float(np.mean(fold_scores))

    def lightgbm_objective(trial):
        parameters = {
            'objective': 'binary', 'n_estimators': 5000,
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 31, 127),
            'min_child_samples': trial.suggest_int('min_child_samples', 30, 180),
            'subsample': trial.suggest_float('subsample', 0.65, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.65, 1.0),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 10.0, log=True),
            'random_state': RANDOM_STATE, 'n_jobs': -1, 'verbosity': -1, **LGBM_DEVICE_PARAMS,
        }
        return mean_fold_auc(lambda: lgb.LGBMClassifier(**parameters), x_train_lgbm)

    def catboost_objective(trial):
        parameters = {
            'loss_function': 'Logloss', 'eval_metric': 'AUC', 'iterations': 5000,
            'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.12, log=True),
            'depth': trial.suggest_int('depth', 5, 10),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 0.1, 20.0, log=True),
            'random_strength': trial.suggest_float('random_strength', 0.01, 2.0, log=True),
            'bootstrap_type': 'Bernoulli',
            'subsample': trial.suggest_float('subsample', 0.65, 1.0),
            'random_seed': RANDOM_STATE, 'allow_writing_files': False, 'verbose': False, **CTB_DEVICE_PARAMS,
        }
        return mean_fold_auc(lambda: ctb.CatBoostClassifier(**parameters), x_train_cat)

    def xgboost_objective(trial):
        parameters = {
            'objective': 'binary:logistic', 'eval_metric': 'auc', 'n_estimators': 5000,
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
            'max_depth': trial.suggest_int('max_depth', 4, 10),
            'min_child_weight': trial.suggest_float('min_child_weight', 1.0, 20.0, log=True),
            'subsample': trial.suggest_float('subsample', 0.65, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.65, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 3.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 10.0, log=True),
            'tree_method': 'hist', 'early_stopping_rounds': 150,
            'random_state': RANDOM_STATE, 'n_jobs': -1, **XGB_DEVICE_PARAMS,
        }
        return mean_fold_auc(lambda: xgb.XGBClassifier(**parameters), x_train_xgb)

    objectives = {
        'lightgbm': lightgbm_objective,
        'catboost': catboost_objective,
        'xgboost': xgboost_objective,
    }
    for model_name in OPTUNA_MODELS:
        if model_name not in objectives:
            raise ValueError(f'Unknown Optuna model: {model_name}')
        study = optuna.create_study(direction='maximize', study_name=f's6e8_{model_name}')
        study.optimize(objectives[model_name], n_trials=N_TRIALS_PER_MODEL, show_progress_bar=True)
        print(f'{model_name} best {N_SPLIT} fold AUC: {study.best_value:.6f}')
        print(study.best_params)


# Experiment 2

In [8]:
is_exp2 = True
if is_exp2:
    lgb_best_params = {
        'learning_rate': 0.019452312132078115, 
        'num_leaves': 31,
        'min_child_samples': 51,
        'subsample': 0.6527230812617075,
        'colsample_bytree': 0.6537711677846225,
        'reg_lambda': 1.5222457391889264
    }

    cbt_best_params = {
        'learning_rate': 0.04752430253765759,
        'depth': 5,
        'l2_leaf_reg': 1.0208188354970549,
        'random_strength': 0.021977846510147718,
        'subsample': 0.9712486515102976
    }

    xgb_best_params = {
        'learning_rate': 0.02282186063326147,
        'max_depth': 4,
        'min_child_weight': 4.124531146139388,
        'subsample': 0.8950114664512429,
        'colsample_bytree': 0.7330021727358563,
        'reg_alpha': 0.21744233498802443,
        'reg_lambda': 0.19941009299776985
    }
    blend_oof_based_on_weight_or_rank(
        'exp2',
        lgbm_params=lgb_best_params,
        ctb_params=cbt_best_params,
        xgb_params=xgb_best_params
    )


[250]	valid_0's auc: 0.939783	valid_0's binary_logloss: 0.232039
[500]	valid_0's auc: 0.940292	valid_0's binary_logloss: 0.229224
[750]	valid_0's auc: 0.940504	valid_0's binary_logloss: 0.22882
[1000]	valid_0's auc: 0.940619	valid_0's binary_logloss: 0.228624
[1250]	valid_0's auc: 0.940693	valid_0's binary_logloss: 0.228494
[1500]	valid_0's auc: 0.940753	valid_0's binary_logloss: 0.228392
[1750]	valid_0's auc: 0.940778	valid_0's binary_logloss: 0.228351
[2000]	valid_0's auc: 0.940806	valid_0's binary_logloss: 0.228304
[2250]	valid_0's auc: 0.9408	valid_0's binary_logloss: 0.228312
LightGBM fold 1: 0.940811 | trees: 2108
[250]	valid_0's auc: 0.940979	valid_0's binary_logloss: 0.230556
[500]	valid_0's auc: 0.94134	valid_0's binary_logloss: 0.227494
[750]	valid_0's auc: 0.941541	valid_0's binary_logloss: 0.227083
[1000]	valid_0's auc: 0.941614	valid_0's binary_logloss: 0.226937
[1250]	valid_0's auc: 0.941675	valid_0's binary_logloss: 0.226822
[1500]	valid_0's auc: 0.941696	valid_0's binar

,OOF ROC AUC
xgboost,0.942056
lightgbm,0.941956
catboost,0.941910


OSError: Cannot save file into a non-existent directory: 'output/1788577914/exp2'